# Classifier Comparison: Significant Terms vs Lindsey's 12 Dimensions

Compares three feature sets for sockpuppet detection on Karin's Pantip data:
  - A) Lindsey's 12-dimension scores (from analyze_comment)
  - B) 34 statistically significant terms as binary features
  - C) Top data-driven n-grams from TF-IDF + chi-squared

All use the same train/test split and same models for fair comparison.

Run AFTER 01_validate_terms_in_karin_dataset.ipynb has confirmed the results.


In [1]:
import re
import json
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2
import warnings
warnings.filterwarnings('ignore')


## SECTION 0: Term Dictionaries (same as validation script)

In [2]:
POLITICAL_TERMS = [
    'ก้าวไกล', 'พท', 'เพื่อไทย', 'ภท', 'ภูมิใจไทย', 'ปชป', 'ประชาธิปัตย์',
    'รทสช', 'รวมไทยสร้างชาติ', 'ชาติไทยพัฒนา', 'ชทพ',
    'พิธา', 'แพทองธาร', 'อุ๊งอิ๊ง', 'เศรษฐา', 'อภิสิทธิ์', 'จุรินทร์',
    'ประยุทธ์', 'ประวิตร', 'พลเอก',
    'เลือกตั้ง', 'นายกรัฐมนตรี', 'นายก', 'ส.ส.', 'สส', 'พรรค',
    'นโยบาย', 'โหวต', 'เขต', 'บัตร', 'หน่วยเลือกตั้ง',
    'รัฐบาล', 'ฝ่ายค้าน', 'รัฐสภา', 'สภา', 'ประชาธิปไตย',
]

ATTACK_TERMS = {
    'โง่': 0.8, 'ไอ้โง่': 0.9, 'ไม่มีสมอง': 0.8, 'ควาย': 0.9,
    'ไอ้ควาย': 1.0, 'โกง': 0.9, 'คอร์รัปชัน': 0.8,
    'สลิ่ม': 0.8, 'แดง': 0.6, 'เหลือง': 0.6, 'เผด็จการ': 0.8,
    'ทหาร': 0.4, 'รัฐประหาร': 0.7, 'ซื้อเสียง': 0.9,
    'ไม่รู้เรื่อง': 0.6, 'พวกนี้': 0.5, 'ไม่เข้าใจ': 0.4,
    'พวกมึง': 0.9, 'มึง': 0.8, 'กู': 0.5,
    'ไอโอ': 0.9, 'io': 0.8, 'บอท': 0.9, 'bot': 0.8,
    'จ้างมา': 0.9, 'รับจ้าง': 0.8, 'ม็อบ': 0.5,
}

EMOTIONAL_TERMS = {
    'ซาบซึ้ง': 0.7, 'ประทับใจ': 0.6, 'ภูมิใจ': 0.7, 'น้ำตาไหล': 0.8,
    'ขนลุก': 0.7, 'อบอุ่น': 0.5, 'ดีใจ': 0.5,
    'น่ากลัว': 0.6, 'หวาดกลัว': 0.7, 'โกรธ': 0.6, 'เกลียด': 0.7,
    'รังเกียจ': 0.7, 'เศร้า': 0.5, 'ผิดหวัง': 0.5,
    'มากๆ': 0.5, 'สุดๆ': 0.6, 'เกินไป': 0.4, 'จริงๆ': 0.3,
    'โคตร': 0.6, 'ห่วย': 0.7, 'แย่มาก': 0.6,
}

LOW_EFFORT_PATTERNS = [
    'เห็นด้วย', 'ถูกต้อง', 'จริง', 'ใช่เลย', 'โอเค', 'ok',
    'ดีมาก', 'เยี่ยม', 'เก่ง', 'สู้ๆ', 'เชียร์',
]

COORDINATION_PHRASES = [
    'มาช่วยกัน', 'ช่วยกันแชร์', 'แชร์ด้วย', 'บอกต่อ',
    'ทุกคนมา', 'ร่วมกัน', '@', 'แท็ก',
]

ALGORITHMIC_TERMS = [
    'แชร์ด้วย', 'ช่วยแชร์', 'กระจายด้วย', 'บอกต่อด้วย',
    'share', 'กด like', 'กดไลค์', 'ไวรัล',
]

AUTHENTICITY_PHRASES = [
    'ปกติไม่ค่อยคอมเมนต์', 'ไม่ค่อยโพสต์', 'เงียบมาตลอด',
    'ครั้งแรกที่', 'ไม่เคยพูดเรื่องการเมือง', 'ในฐานะที่',
    'ขอพูดตรงๆ', 'พูดตามตรง',
]

# The 34 statistically significant terms from validation (p < 0.05)
# with their direction (SOCK↑ or NON-SOCK↑)
SIGNIFICANT_TERMS = {
    # SOCK↑ terms (more frequent in sockpuppet posts)
    'ก้าวไกล': 'SOCK', 'ม็อบ': 'SOCK', 'โง่': 'SOCK', 'แดง': 'SOCK',
    'นโยบาย': 'SOCK', 'เลือกตั้ง': 'SOCK', 'พิธา': 'SOCK',
    'ประยุทธ์': 'SOCK', 'พรรค': 'SOCK', 'เพื่อไทย': 'SOCK',
    'พลเอก': 'SOCK', 'รัฐประหาร': 'SOCK', 'ส.ส.': 'SOCK',
    'กู': 'SOCK', 'ซื้อเสียง': 'SOCK', 'เผด็จการ': 'SOCK',
    'สลิ่ม': 'SOCK', 'ทหาร': 'SOCK', 'เหลือง': 'SOCK',
    'จ้างมา': 'SOCK', 'สส': 'SOCK', 'ประชาธิปัตย์': 'SOCK',
    'รัฐบาล': 'SOCK', 'สภา': 'SOCK', 'ok': 'SOCK', 'เชียร์': 'SOCK',
    # NON-SOCK↑ terms (more frequent in non-sockpuppet posts)
    'มากๆ': 'NON-SOCK', 'จริง': 'NON-SOCK', 'บัตร': 'NON-SOCK',
    'สู้ๆ': 'NON-SOCK', 'พท': 'NON-SOCK', 'เกินไป': 'NON-SOCK',
    'เศร้า': 'NON-SOCK', 'ผิดหวัง': 'NON-SOCK',
}


## SECTION 1: Load Data (same logic as validation script)

In [3]:
def load_karin_dataset(data_path: str) -> pd.DataFrame:
    if data_path.endswith('.csv'):
        df = pd.read_csv(data_path, encoding='utf-8-sig')
    else:
        with open(data_path, 'r', encoding='utf-8-sig', errors='replace') as f:
            raw = f.read()
        try:
            data = json.loads(raw)
        except json.JSONDecodeError as e:
            print(f"[WARNING] Standard JSON failed: {e}")
            print("[WARNING] Attempting repair (truncation)...")
            truncated = raw[:e.pos]
            last_brace = truncated.rfind('}')
            if last_brace > 0:
                truncated = truncated[:last_brace + 1].rstrip().rstrip(',') + '\n]'
                data = json.loads(truncated)
                print(f"[WARNING] Loaded {len(data)} records (truncated)")
            else:
                raise
        df = pd.DataFrame(data)

    print(f"Loaded {len(df)} posts")

    sock = (df['is_sockpuppet'] == 1).sum()
    non_sock = (df['is_sockpuppet'] == 0).sum()
    print(f"Labels: sockpuppet={sock}, non-sockpuppet={non_sock}")
    print(f"Unique users: {df['user_id'].nunique()}")

    return df

## SECTION 2: Feature Extraction — Three Approaches

In [4]:
def extract_lindsey_12dim(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature Set A: Lindsey's 12 behavioral dimension scores.
    Same analyze_comment logic from the validation script.
    """
    emoji_pat = re.compile(
        r'[\U0001F600-\U0001F64F]|[\U0001F300-\U0001F5FF]'
        r'|[\U0001F680-\U0001F6FF]|[\U0001F1E0-\U0001F1FF]'
        r'|[\U00002600-\U000026FF]|[\U00002700-\U000027BF]'
    )

    all_attack = list(ATTACK_TERMS.keys())
    all_emotional = list(EMOTIONAL_TERMS.keys())

    records = []
    for text in df['messages'].fillna(''):
        if not text or not isinstance(text, str):
            text = ''
        text = text.strip()
        lower = text.lower()
        emojis = emoji_pat.findall(text)
        text_no_emoji = emoji_pat.sub('', text)
        words = text_no_emoji.split()
        wc = len(words)
        ec = len(emojis)

        # emoji
        if ec == 0: e_s = 0.0
        elif ec <= 2: e_s = 0.2
        elif ec <= 4: e_s = 0.4
        elif ec <= 6: e_s = 0.6
        elif ec <= 9: e_s = 0.8
        else: e_s = 1.0

        # political
        pc = sum(1 for t in POLITICAL_TERMS if t in lower)
        p_s = min(1.0, pc / max(1, wc) * 5)

        # attack
        a_s = max((ATTACK_TERMS[t] for t in all_attack if t in lower), default=0.0)

        # emotional
        em_s = max((EMOTIONAL_TERMS[t] for t in all_emotional if t in lower), default=0.0)

        # low effort
        le_s = 0.0
        if wc <= 3:
            for p in LOW_EFFORT_PATTERNS:
                if p in lower: le_s = 0.8; break

        # coordination
        co_s = 0.0
        for p in COORDINATION_PHRASES:
            if p in lower: co_s = max(co_s, 0.4)

        # algorithmic
        al_s = 0.0
        for t in ALGORITHMIC_TERMS:
            if t in lower: al_s = max(al_s, 0.7)

        # authenticity
        au_s = 0.0
        for p in AUTHENTICITY_PHRASES:
            if p in lower: au_s = 0.8; break

        # length
        if wc == 0: ln_s = 0.6
        elif wc <= 2: ln_s = 0.4
        elif wc >= 50: ln_s = 0.3
        else: ln_s = 0.0

        # repetition
        rp_s = 0.0
        if re.search(r'[!]{3,}', text) or re.search(r'[?]{3,}', text): rp_s = 0.4
        if re.search(r'(.)\1{3,}', lower): rp_s = max(rp_s, 0.3)

        # punctuation
        pn_count = len(re.findall(r'[!?.,;:]', text))
        if wc > 0:
            pr = pn_count / wc
            pn_s = 0.6 if pr > 0.5 else (0.3 if pr > 0.3 else 0.0)
        else:
            pn_s = 0.0

        # caps
        caps = sum(1 for c in text if c.isupper())
        if len(text) > 0:
            cr = caps / len(text)
            cp_s = 0.8 if (cr > 0.7 and len(text) > 10) else (0.4 if cr > 0.5 else 0.0)
        else:
            cp_s = 0.0

        records.append({
            'emoji_score': e_s, 'political_score': p_s, 'attack_score': a_s,
            'emotional_score': em_s, 'low_effort_score': le_s,
            'coordination_score': co_s, 'algorithmic_score': al_s,
            'authenticity_score': au_s, 'length_score': ln_s,
            'repetition_score': rp_s, 'punctuation_score': pn_s,
            'caps_score': cp_s,
        })

    return pd.DataFrame(records)



In [5]:
def extract_significant_terms(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature Set B: 34 statistically significant terms as binary features.
    Each column = 1 if term present in post, 0 otherwise.
    """
    texts = df['messages'].fillna('').str.lower()
    records = {}
    for term in SIGNIFICANT_TERMS:
        col_name = f"term_{term}"
        records[col_name] = texts.str.contains(re.escape(term), na=False).astype(int)

    return pd.DataFrame(records)


def extract_datadriven_features(df: pd.DataFrame, max_features: int = 200) -> tuple:
    """
    Feature Set C: TF-IDF character n-grams selected by chi-squared.
    Returns (feature_matrix, feature_names).
    """
    texts = df['messages'].fillna('').tolist()
    labels = df['is_sockpuppet'].values

    vectorizer = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 6),
        min_df=5,
        max_df=0.5,
        max_features=5000,
    )
    X = vectorizer.fit_transform(texts)
    feature_names = vectorizer.get_feature_names_out()

    # Select top features by chi-squared
    chi2_scores, p_values = chi2(X, labels)
    top_indices = np.argsort(chi2_scores)[-max_features:]
    X_selected = X[:, top_indices]
    selected_names = feature_names[top_indices]

    return X_selected, selected_names, vectorizer


## SECTION 3: Run Classification Experiments

In [6]:
def run_experiment(X, y, feature_set_name: str, train_idx, test_idx,
                   feature_names=None, groups_train=None):
    """
    Train/test with multiple classifiers, report metrics.
    Uses pre-computed user-level train/test split.
    CV uses GroupKFold by user_id on training set.
    """
    print(f"\n{'='*70}")
    print(f"  FEATURE SET: {feature_set_name}")
    print(f"  Shape: {X.shape[0]} samples × {X.shape[1]} features")
    print(f"  Train: {len(train_idx)} posts, Test: {len(test_idx)} posts")
    print(f"{'='*70}")

    if hasattr(X, 'toarray'):  # sparse matrix
        X_train = X[train_idx]
        X_test = X[test_idx]
    elif isinstance(X, np.ndarray):
        X_train = X[train_idx]
        X_test = X[test_idx]
    else:
        X_train = X.iloc[train_idx].values if hasattr(X, 'iloc') else X[train_idx]
        X_test = X.iloc[test_idx].values if hasattr(X, 'iloc') else X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    models = {
        'Random Forest': RandomForestClassifier(
            n_estimators=200, max_depth=20, random_state=42, n_jobs=-1
        ),
        'Gradient Boosting': GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42
        ),
        'Logistic Regression': LogisticRegression(
            max_iter=1000, random_state=42, C=1.0
        ),
    }

    results = []

    for name, model in models.items():
        # Scale for logistic regression
        if name == 'Logistic Regression':
            scaler = StandardScaler(with_mean=False)
            X_train_use = scaler.fit_transform(X_train)
            X_test_use = scaler.transform(X_test)
        else:
            X_train_use = X_train
            X_test_use = X_test

        # Train
        model.fit(X_train_use, y_train)
        y_pred = model.predict(X_test_use)

        # Probabilities for AUC
        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(X_test_use)[:, 1]
            auc = roc_auc_score(y_test, y_prob)
        else:
            auc = 0.0

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        # 5-fold GroupKFold CV on training set (grouped by user)
        if groups_train is not None:
            from sklearn.model_selection import GroupKFold
            gkf = GroupKFold(n_splits=5)
            cv_scores = cross_val_score(
                model, X_train_use, y_train,
                cv=gkf, scoring='f1', groups=groups_train
            )
        else:
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            cv_scores = cross_val_score(model, X_train_use, y_train, cv=cv, scoring='f1')

        result = {
            'feature_set': feature_set_name,
            'model': name,
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'f1': f1,
            'auc': auc,
            'cv_f1_mean': cv_scores.mean(),
            'cv_f1_std': cv_scores.std(),
        }
        results.append(result)

        print(f"\n  {name}:")
        print(f"    Accuracy:  {acc:.4f}")
        print(f"    Precision: {prec:.4f}")
        print(f"    Recall:    {rec:.4f}")
        print(f"    F1:        {f1:.4f}")
        print(f"    AUC:       {auc:.4f}")
        print(f"    CV F1:     {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

        # Feature importance for tree-based models
        if feature_names is not None and hasattr(model, 'feature_importances_'):
            importances = model.feature_importances_
            top_idx = np.argsort(importances)[-10:][::-1]
            fn = feature_names if not hasattr(feature_names, 'tolist') else feature_names.tolist()
            print(f"    Top 10 features:")
            for i in top_idx:
                if i < len(fn):
                    print(f"      {fn[i]:<30} {importances[i]:.4f}")

    return results

## SECTION 4: Comparison Summary

In [7]:
def print_comparison(all_results):
    """Print side-by-side comparison of all feature sets."""

    print("\n" + "=" * 90)
    print("  FINAL COMPARISON: All Feature Sets × All Models")
    print("=" * 90)

    df = pd.DataFrame(all_results)

    # Best model per feature set
    print("\n── Best F1 per Feature Set ──")
    print(f"{'Feature Set':<35} {'Model':<22} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7}")
    print("-" * 90)

    for fs in df['feature_set'].unique():
        fs_df = df[df['feature_set'] == fs]
        best = fs_df.loc[fs_df['f1'].idxmax()]
        print(f"{best['feature_set']:<35} {best['model']:<22} "
              f"{best['accuracy']:>7.4f} {best['precision']:>7.4f} "
              f"{best['recall']:>7.4f} {best['f1']:>7.4f} {best['auc']:>7.4f}")

    # Full table
    print("\n── Full Results ──")
    print(f"{'Feature Set':<35} {'Model':<22} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7} {'CV F1':>12}")
    print("-" * 105)

    for _, row in df.iterrows():
        cv_str = f"{row['cv_f1_mean']:.4f}±{row['cv_f1_std']:.4f}"
        print(f"{row['feature_set']:<35} {row['model']:<22} "
              f"{row['accuracy']:>7.4f} {row['precision']:>7.4f} "
              f"{row['recall']:>7.4f} {row['f1']:>7.4f} {row['auc']:>7.4f} "
              f"{cv_str:>12}")

    # Save
    df.to_csv('classifier_comparison.csv', index=False)
    print("\nSaved: classifier_comparison.csv")

## Main - CONFIG & RUN

In [8]:
DATA_PATH = '00_Datasets_After_preprocessed.csv'   # ← your Karin dataset path

# Load
df = load_karin_dataset(DATA_PATH)
y = df['is_sockpuppet'].values
user_ids = df['user_id'].values

# ── User-level train/test split ──
# All posts from a user go entirely into train OR test (no leakage)
unique_users = df[['user_id', 'is_sockpuppet']].drop_duplicates('user_id')
train_users, test_users = train_test_split(
    unique_users['user_id'].values,
    test_size=0.2,
    random_state=42,
    stratify=unique_users['is_sockpuppet'].values
)
train_user_set = set(train_users)
test_user_set = set(test_users)

train_idx = np.where(df['user_id'].isin(train_user_set))[0]
test_idx = np.where(df['user_id'].isin(test_user_set))[0]

# Groups for GroupKFold CV (user_id per training post)
groups_train = user_ids[train_idx]

print(f"\nUser-level split:")
print(f"  Train: {len(train_users)} users, {len(train_idx)} posts")
print(f"  Test:  {len(test_users)} users, {len(test_idx)} posts")
print(f"  Train sockpuppet posts: {y[train_idx].sum()} / {len(train_idx)}")
print(f"  Test  sockpuppet posts: {y[test_idx].sum()} / {len(test_idx)}")

all_results = []

# ── Feature Set A: Lindsey's 12 Dimensions ──
print("\n\nExtracting Feature Set A: Lindsey's 12 Dimensions...")
X_lindsey = extract_lindsey_12dim(df)
lindsey_names = list(X_lindsey.columns)
results_a = run_experiment(
    X_lindsey.values, y, "A: Lindsey 12 Dimensions",
    train_idx, test_idx, lindsey_names, groups_train
)
all_results.extend(results_a)

# ── Feature Set B: 19 Significant Terms (binary) ──
print("\n\nExtracting Feature Set B: 19 Significant Terms...")
X_terms = extract_significant_terms(df)
term_names = list(X_terms.columns)
results_b = run_experiment(
    X_terms.values, y, "B: 19 Significant Terms",
    train_idx, test_idx, term_names, groups_train
)
all_results.extend(results_b)

# ── Feature Set C: Data-Driven N-grams ──
print("\n\nExtracting Feature Set C: Data-Driven N-grams (top 200)...")
X_ngrams, ngram_names, vectorizer = extract_datadriven_features(df, max_features=200)
results_c = run_experiment(
    X_ngrams, y, "C: Data-Driven N-grams (200)",
    train_idx, test_idx, ngram_names.tolist(), groups_train
)
all_results.extend(results_c)

# ── Feature Set D: Significant Terms + Data-Driven (combined) ──
print("\n\nExtracting Feature Set D: Combined (19 terms + top 200 n-grams)...")
import scipy.sparse as sp
X_terms_sparse = sp.csr_matrix(X_terms.values)
X_combined = sp.hstack([X_terms_sparse, X_ngrams])
combined_names = term_names + ngram_names.tolist()
results_d = run_experiment(
    X_combined, y, "D: Combined (terms + n-grams)",
    train_idx, test_idx, combined_names, groups_train
)
all_results.extend(results_d)

# ── Print Comparison ──
print_comparison(all_results)

Loaded 5457 posts
Labels: sockpuppet=2390, non-sockpuppet=3067
Unique users: 394

User-level split:
  Train: 315 users, 4097 posts
  Test:  79 users, 1360 posts
  Train sockpuppet posts: 1924 / 4097
  Test  sockpuppet posts: 466 / 1360


Extracting Feature Set A: Lindsey's 12 Dimensions...

  FEATURE SET: A: Lindsey 12 Dimensions
  Shape: 5457 samples × 12 features
  Train: 4097 posts, Test: 1360 posts

  Random Forest:
    Accuracy:  0.6110
    Precision: 0.3960
    Recall:    0.2575
    F1:        0.3121
    AUC:       0.5129
    CV F1:     0.3467 ± 0.0494
    Top 10 features:
      political_score                0.4568
      attack_score                   0.1837
      emotional_score                0.1250
      punctuation_score              0.0605
      length_score                   0.0603
      repetition_score               0.0430
      emoji_score                    0.0250
      low_effort_score               0.0222
      coordination_score             0.0197
      authenticity